# Genetic Algorithms - Tutorial 3: Advanced Topics and Applications

Welcome to Tutorial 3! This tutorial covers advanced GA applications:

## Topics:

1. **Traveling Salesman Problem (TSP)** - Combinatorial optimization
2. **Permutation Encoding** - Special operators for ordering problems
3. **Hybrid Algorithms** - GA + Local Search
4. **Multi-Objective Optimization** - Introduction to NSGA-II concepts
5. **Real-World Applications** - Putting it all together

**Prerequisites:** Tutorials 1 and 2

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys

# Import utilities
sys.path.append('../GA_Tutorial_1_Basics')
sys.path.append('../GA_Tutorial_2_Intermediate')

from ga_utils_basics import *
from ga_utils_intermediate import *
from ga_utils_advanced import *

np.random.seed(42)
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Imports successful!")

## Part 1: Traveling Salesman Problem (TSP)

### What is TSP?

**Problem:** Find the shortest route visiting all cities exactly once and returning to start.

**Why challenging?**
- Number of possible tours: (n-1)!/2 
- For 20 cities: 60,822,550,204,416,000 possible tours!
- NP-hard problem

**GA Advantages:**
- Can find good solutions quickly
- Doesn't require exact optimum
- Works for large instances

### TSP Representation

**Permutation Encoding:**
```
Tour: [0, 3, 1, 4, 2]
Meaning: Start at city 0 → city 3 → city 1 → city 4 → city 2 → back to 0
```

In [ ]:
# Create a small TSP instance
print("="*70)
print("Creating TSP Instance")
print("="*70)

n_cities = 10
cities, distance_matrix = create_symmetric_tsp(n_cities)

print(f"\nNumber of cities: {n_cities}")
print(f"Possible tours: {np.math.factorial(n_cities-1) // 2:,}")
print(f"\nCity coordinates:")
for i, city in enumerate(cities[:5]):  # Show first 5
    print(f"City {i}: ({city[0]:.2f}, {city[1]:.2f})")
print("...")

# Example tour
example_tour = np.arange(n_cities)
example_distance = calculate_tsp_distance(example_tour, distance_matrix)
print(f"\nExample tour {example_tour[:5]}... distance: {example_distance:.2f}")

# Visualize
plot_tsp_tour(cities, example_tour, title="TSP Instance - Sequential Tour")

### TSP-Specific Operators

**Standard operators DON'T work for permutations!**

❌ Single-point crossover:
```
Parent 1: [0,1,2 | 3,4]
Parent 2: [4,3,2 | 1,0]
          -------
Offspring: [0,1,2 | 1,0]  ← Invalid! City 1 appears twice, city 3,4 missing
```

✅ **Order Crossover (OX):** Preserves relative order

✅ **PMX (Partially Mapped Crossover):** Uses mapping to fix conflicts

✅ **Swap/Inversion Mutation:** Maintains permutation validity

In [ ]:
# Test Order Crossover
print("="*70)
print("Order Crossover (OX) Demonstration")
print("="*70)

parent1 = np.array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
parent2 = np.array([9, 8, 7, 6, 5, 4, 3, 2, 1, 0])

print(f"\nParent 1: {parent1}")
print(f"Parent 2: {parent2}")

np.random.seed(42)
off1, off2 = order_crossover(parent1, parent2)

print(f"\nOffspring 1: {off1}")
print(f"Offspring 2: {off2}")

# Verify validity
print(f"\nOffspring 1 valid? {len(set(off1)) == len(off1) and set(off1) == set(parent1)}")
print(f"Offspring 2 valid? {len(set(off2)) == len(off2) and set(off2) == set(parent2)}")

In [ ]:
# Test TSP Mutations
print("="*70)
print("TSP Mutation Operators")
print("="*70)

tour = np.array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
print(f"\nOriginal tour: {tour}")

# Swap mutation
np.random.seed(10)
swapped = swap_mutation(tour, mutation_rate=1.0)
print(f"\nSwap mutation: {swapped}")

# Inversion mutation
np.random.seed(20)
inverted = inversion_mutation(tour, mutation_rate=1.0)
print(f"Inversion mutation: {inverted}")

# Scramble mutation
np.random.seed(30)
scrambled = scramble_mutation(tour, mutation_rate=1.0)
print(f"Scramble mutation: {scrambled}")

print("\n✓ All mutations preserve permutation validity!")

### Complete TSP Genetic Algorithm

In [ ]:
def tsp_genetic_algorithm(distance_matrix, pop_size=100, max_generations=500,
                          mutation_rate=0.1, elite_size=2, verbose=True):
    """
    Genetic Algorithm for TSP.
    
    Arguments:
    distance_matrix -- distances between cities
    pop_size -- population size
    max_generations -- max generations
    mutation_rate -- mutation probability
    elite_size -- number of elites
    verbose -- print progress
    
    Returns:
    best_tour, best_distance, history
    """
    n_cities = len(distance_matrix)
    
    # Initialize population
    population = initialize_tsp_population(pop_size, n_cities)
    
    history = {'best': [], 'average': [], 'worst': []}
    
    for generation in range(max_generations):
        
        # Evaluate fitness (negative distance for maximization)
        distances = np.array([calculate_tsp_distance(tour, distance_matrix) for tour in population])
        fitness = -distances
        
        # Track statistics
        history['best'].append(-np.max(fitness))
        history['average'].append(-np.mean(fitness))
        history['worst'].append(-np.min(fitness))
        
        if verbose and (generation % 100 == 0 or generation == max_generations - 1):
            print(f"Gen {generation:3d} | Best distance: {-np.max(fitness):.2f} | "
                  f"Avg: {-np.mean(fitness):.2f}")
        
        # Selection (tournament)
        parents = tournament_selection(population, fitness, pop_size, tournament_size=3)
        
        # Create offspring
        offspring = []
        for i in range(0, pop_size, 2):
            p1 = parents[i]
            p2 = parents[min(i+1, pop_size-1)]
            
            # Crossover
            if np.random.rand() < 0.9:
                c1, c2 = order_crossover(p1, p2)
            else:
                c1, c2 = p1.copy(), p2.copy()
            
            # Mutation
            c1 = inversion_mutation(c1, mutation_rate)
            c2 = inversion_mutation(c2, mutation_rate)
            
            offspring.append(c1)
            if len(offspring) < pop_size:
                offspring.append(c2)
        
        population = np.array(offspring[:pop_size])
        
        # Elitism
        if elite_size > 0:
            elite_indices = np.argsort(fitness)[-elite_size:]
            for i, elite_idx in enumerate(elite_indices):
                population[i] = parents[elite_idx]
    
    # Final evaluation
    distances = np.array([calculate_tsp_distance(tour, distance_matrix) for tour in population])
    best_idx = np.argmin(distances)
    
    return population[best_idx], distances[best_idx], history

print("✓ TSP GA implementation complete!")

In [ ]:
# Run TSP GA
print("="*70)
print("Running TSP Genetic Algorithm")
print("="*70)

np.random.seed(42)
n_cities = 20
cities, distance_matrix = create_symmetric_tsp(n_cities)

best_tour, best_distance, history = tsp_genetic_algorithm(
    distance_matrix,
    pop_size=100,
    max_generations=500,
    mutation_rate=0.1,
    elite_size=2,
    verbose=True
)

print(f"\n{'='*70}")
print("FINAL RESULT")
print(f"{'='*70}")
print(f"Best tour distance: {best_distance:.2f}")
print(f"Best tour (first 10 cities): {best_tour[:10]}...")

In [ ]:
# Visualize results
plot_tsp_tour(cities, best_tour, title=f"Best TSP Tour (distance={best_distance:.2f})")

# Plot convergence
plt.figure(figsize=(10, 5))
plt.plot(history['best'], label='Best', linewidth=2)
plt.plot(history['average'], label='Average', linewidth=1.5, linestyle='--')
plt.xlabel('Generation')
plt.ylabel('Tour Distance')
plt.title('TSP GA Convergence')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Part 2: Hybrid Algorithms - GA + Local Search

**Problem:** GAs are good at global search but slow at local refinement.

**Solution:** Combine with local search!

### Types:

1. **Lamarckian Evolution**
   - Apply local search
   - **Modify genotype** with improved solution
   - Learned improvements are inherited

2. **Baldwin Effect**
   - Apply local search
   - **Don't modify genotype**
   - Use improvement for fitness only
   - Maintains genetic diversity

### 2-opt Local Search

For TSP, repeatedly try edge swaps:
```
Original: A→B→C→D→E
2-opt:    A→B→D→C→E  (reverse C-D edge)
Keep if improved!
```

In [ ]:
# Demonstrate 2-opt improvement
print("="*70)
print("2-opt Local Search Demonstration")
print("="*70)

# Create random tour
random_tour = np.random.permutation(n_cities)
initial_distance = calculate_tsp_distance(random_tour, distance_matrix)

print(f"\nInitial tour distance: {initial_distance:.2f}")

# Apply 2-opt
improved_tour = two_opt_local_search(random_tour, distance_matrix, max_iterations=100)
improved_distance = calculate_tsp_distance(improved_tour, distance_matrix)

print(f"Improved tour distance: {improved_distance:.2f}")
print(f"Improvement: {initial_distance - improved_distance:.2f} ({(initial_distance-improved_distance)/initial_distance*100:.1f}%)")

In [ ]:
# Hybrid GA with Lamarckian evolution
def hybrid_tsp_ga(distance_matrix, pop_size=100, max_generations=300,
                  local_search_prob=0.2, verbose=True):
    """
    Hybrid GA for TSP with 2-opt local search (Lamarckian).
    """
    n_cities = len(distance_matrix)
    population = initialize_tsp_population(pop_size, n_cities)
    history = {'best': [], 'average': []}
    
    for generation in range(max_generations):
        # Evaluate
        distances = np.array([calculate_tsp_distance(tour, distance_matrix) for tour in population])
        fitness = -distances
        
        history['best'].append(-np.max(fitness))
        history['average'].append(-np.mean(fitness))
        
        if verbose and (generation % 100 == 0 or generation == max_generations - 1):
            print(f"Gen {generation:3d} | Best: {-np.max(fitness):.2f}")
        
        # Selection
        parents = tournament_selection(population, fitness, pop_size, tournament_size=3)
        
        # Crossover & Mutation
        offspring = []
        for i in range(0, pop_size, 2):
            c1, c2 = order_crossover(parents[i], parents[min(i+1, pop_size-1)])
            c1 = inversion_mutation(c1, 0.1)
            c2 = inversion_mutation(c2, 0.1)
            offspring.extend([c1, c2])
        
        population = np.array(offspring[:pop_size])
        
        # LAMARCKIAN: Apply local search and modify genotype
        population = lamarckian_evolution(population, distance_matrix, local_search_prob)
    
    distances = np.array([calculate_tsp_distance(tour, distance_matrix) for tour in population])
    best_idx = np.argmin(distances)
    return population[best_idx], distances[best_idx], history

print("✓ Hybrid GA implementation complete!")

In [ ]:
# Compare standard vs hybrid GA
print("="*70)
print("COMPARISON: Standard GA vs Hybrid GA")
print("="*70)

np.random.seed(42)
cities_test, dist_matrix_test = create_symmetric_tsp(15)

# Standard GA
print("\nRunning Standard GA...")
np.random.seed(42)
_, dist_standard, hist_standard = tsp_genetic_algorithm(
    dist_matrix_test, pop_size=50, max_generations=200, verbose=False
)

# Hybrid GA
print("Running Hybrid GA (with 2-opt)...")
np.random.seed(42)
_, dist_hybrid, hist_hybrid = hybrid_tsp_ga(
    dist_matrix_test, pop_size=50, max_generations=200, 
    local_search_prob=0.3, verbose=False
)

print(f"\n{'='*70}")
print("RESULTS")
print(f"{'='*70}")
print(f"Standard GA final distance: {dist_standard:.2f}")
print(f"Hybrid GA final distance:   {dist_hybrid:.2f}")
print(f"Improvement: {dist_standard - dist_hybrid:.2f} ({(dist_standard-dist_hybrid)/dist_standard*100:.1f}%)")

# Plot comparison
plt.figure(figsize=(10, 5))
plt.plot(hist_standard['best'], label='Standard GA', linewidth=2)
plt.plot(hist_hybrid['best'], label='Hybrid GA (with 2-opt)', linewidth=2)
plt.xlabel('Generation')
plt.ylabel('Best Distance')
plt.title('Standard vs Hybrid GA Convergence')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Part 3: Multi-Objective Optimization Basics

Real-world problems often have **multiple conflicting objectives**:

- **Cost vs Quality**
- **Speed vs Accuracy**
- **Performance vs Energy**

### Pareto Dominance

Solution A **dominates** B if:
- A is no worse than B in all objectives
- A is strictly better than B in at least one objective

### Pareto Front

Set of **non-dominated** solutions - tradeoff curve.

### Example: Minimize both f1 and f2

```
Solution A: (f1=2, f2=5)  ← Dominates B
Solution B: (f1=3, f2=6)  
Solution C: (f1=1, f2=7)  ← Non-dominated (better f1, worse f2)
```

In [ ]:
# Example: Two-objective problem
print("="*70)
print("Multi-Objective Optimization Example")
print("="*70)

# Generate random solutions
np.random.seed(42)
n_solutions = 100
objectives = np.random.uniform(0, 10, size=(n_solutions, 2))

print(f"\nGenerated {n_solutions} random solutions with 2 objectives")
print(f"Sample solutions:")
for i in range(5):
    print(f"Solution {i}: obj1={objectives[i,0]:.2f}, obj2={objectives[i,1]:.2f}")

# Perform non-dominated sorting
dummy_pop = np.arange(n_solutions).reshape(-1, 1)
fronts = fast_non_dominated_sort(dummy_pop, objectives)

print(f"\nNon-dominated sorting results:")
for i, front in enumerate(fronts[:5]):
    print(f"Front {i+1}: {len(front)} solutions")

# Visualize Pareto fronts
plot_pareto_front_2d(objectives, title="Multi-Objective Solutions - Pareto Fronts")

### NSGA-II Key Concepts

**NSGA-II** (Non-dominated Sorting Genetic Algorithm II) is the most popular multi-objective GA.

**Two mechanisms:**

1. **Non-dominated sorting** - Rank solutions by dominance
2. **Crowding distance** - Prefer solutions in less crowded regions

**Selection:**
- Prefer lower rank (better front)
- Within same rank, prefer higher crowding distance (more isolated)

This tutorial provides the foundation - full NSGA-II is more complex!

In [ ]:
# Demonstrate crowding distance
print("="*70)
print("Crowding Distance for Diversity")
print("="*70)

# Use first front
first_front_indices = fronts[0]
first_front_objectives = objectives[first_front_indices]

# Calculate crowding distances
crowding_dists = calculate_crowding_distance_multi(first_front_objectives)

print(f"\nFirst Pareto front has {len(first_front_indices)} solutions")
print(f"\nCrowding distances (sorted):")

sorted_indices = np.argsort(crowding_dists)[::-1]
for i in sorted_indices[:5]:
    dist_str = "∞" if np.isinf(crowding_dists[i]) else f"{crowding_dists[i]:.4f}"
    print(f"Solution {first_front_indices[i]}: distance = {dist_str}")

print("\n💡 Higher crowding distance = more isolated = preferred for diversity")

## Conclusion

### What You've Learned

✅ **TSP and Permutation Encoding**
- Order crossover (OX)
- Partially mapped crossover (PMX)
- Swap, inversion, scramble mutations
- Complete TSP GA implementation

✅ **Hybrid Algorithms**
- 2-opt local search
- Lamarckian evolution
- Baldwin effect
- GA + local search integration

✅ **Multi-Objective Basics**
- Pareto dominance
- Non-dominated sorting
- Crowding distance
- NSGA-II foundations

### Key Takeaways

1. **Encoding matters**: Use appropriate representation (permutation for TSP)
2. **Hybrid > Pure GA**: Local search dramatically improves results
3. **Multiple objectives**: Need special handling (Pareto fronts)
4. **Problem-specific operators**: Design operators for your problem

### Next: Tutorial 4

Real-world applications:
- Hyperparameter optimization
- Feature selection
- Neural architecture search
- Production scheduling

---

**Congratulations!** You now have advanced GA skills for complex real-world problems! 🎉